# `semantic.v_measure_summary` — view

Thin view over Gold. No logic beyond shaping.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [0]:
-- THE SUMMARY. One row per horizon and measure: where the index landed, what the best
-- trust did, and how many beat it. Every counter on the dashboard reads a column of this.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_measure_summary
COMMENT 'One row per horizon and measure: where the index lands, and what the best trust did'
AS
WITH ranked AS (
  SELECT horizon_years, measure, ticker, trust_name, value, value_pct,
         ROW_NUMBER() OVER (PARTITION BY horizon_years, measure ORDER BY value DESC) AS rn
  FROM `index-vs-trust-pipeline`.semantic.v_measure
  WHERE entity_type = 'Trust'
),
best AS (
  -- The single strongest trust on this measure, named as well as measured.
  SELECT horizon_years, measure, ticker AS best_ticker, trust_name AS best_name,
         value_pct AS best_pct
  FROM ranked WHERE rn = 1
)
SELECT m.horizon_years,
       m.measure,
       MAX(CASE WHEN m.ticker = 'SPY' THEN m.value_pct END)        AS index_pct,
       MAX(CASE WHEN m.ticker = 'SPY' THEN m.rank_in_measure END)  AS index_rank,
       MAX(b.best_pct)                                             AS best_trust_pct,
       MAX(b.best_ticker)                                          AS best_trust_ticker,
       MAX(b.best_name)                                            AS best_trust_name,
       -- A counter shows one value, so the name and the number travel together.
       CONCAT(MAX(b.best_ticker), '  ',
              FORMAT_NUMBER(MAX(b.best_pct), 0), '%')              AS best_trust_label,
       COUNT(CASE WHEN m.entity_type = 'Trust' THEN 1 END)         AS trusts,
       SUM(CASE WHEN m.entity_type = 'Trust' AND m.beats_index
                THEN 1 ELSE 0 END)                                 AS trusts_above,
       ROUND(100.0 * SUM(CASE WHEN m.entity_type = 'Trust' AND m.beats_index
                              THEN 1 ELSE 0 END)
             / COUNT(CASE WHEN m.entity_type = 'Trust' THEN 1 END), 1) AS pct_above
FROM `index-vs-trust-pipeline`.semantic.v_measure m
JOIN best b ON b.horizon_years = m.horizon_years AND b.measure = m.measure
GROUP BY m.horizon_years, m.measure;

## Verification

Expected: the view resolves and returns rows. Counts are in 
`
specs/04_semantic/dashboard.md
`
.

In [0]:
SELECT COUNT(*) AS rows
FROM `index-vs-trust-pipeline`.semantic.v_measure_summary;